### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel  ##same like transformer library of hugging face
import torch
max_seq_length = 2048 ## now keepin it 2048 later will check on my dataset max token length model will process
dtype = None ##unsloth will auto decide the datatype of numbers
load_in_4bit = True # Use 4bit quantization to reduce memory usage. To reduce GPU memory usage while keeping model quality high during QLoRA fine-tuning.

model, tokenizer = FastLanguageModel.from_pretrained(
  #using instruct model
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,

)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
# Attach LoRA adapters to selected transformer layers.
# Only the adapters are trained, while the original Qwen weights remain frozen.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, #capacity of lora adapters
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], # integrate lora to each and every layer of model
    lora_alpha = 16, #strenght scaling of lora updaters
    lora_dropout = 0, #remove unnecessary dead neurons
    bias = "none",  # extra weight

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.7.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
train_dataset=load_dataset(
    "json",
    data_files="train.json",
    split="train"
)
validation_dataset = load_dataset(
    "json",
    data_files="validation.json",
    split="train",
)
SYSTEM_PROMPT = (
    "You are an expert AI/ML Professor teaching computer science students at a top Indian engineering college.\n\n"

    "Your task is to explain AI/ML research papers in a simple, educational, and technically accurate manner.\n\n"

    "Rules:\n"
    "1. Write in natural educational Hinglish using ONLY the Roman (Latin) script. Never use Devanagari.\n"
    "2. Maintain a professional classroom teaching style. Avoid slang or casual internet language.\n"
    "3. Keep all AI/ML technical terms in English (e.g., Transformer, Gradient Descent, Backpropagation, Loss Function, Attention, Embedding).\n"
    "4. Explain concepts with intuition first before mentioning technical details whenever appropriate.\n"
    "5. Base every explanation only on the provided research paper context. Never invent results, datasets, experiments, or metrics. If information is missing, clearly state that it is not mentioned in the provided context.\n"
    "6. Keep explanations well-structured with short paragraphs for readability."
)

def formatting_prompts_func(examples):
    texts = []

    for instruction, inp, output in zip(
        examples["instruction"],
        examples["input"],
        examples["output"],
    ):
        messages = [  #auto decided the chatml format
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },

            {
                  "role": "user",
                  "content": (
              f"Task:\n{instruction}\n\n"
              f"Research Paper Context:\n{inp}"
              ),
            },

            {
                "role": "assistant",
                "content": output,
            },
        ]

        text = tokenizer.apply_chat_template(  #here apply_chat_template converts direclty to chatml format req for qwen model
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        texts.append(text)

    return {"text": texts}


train_dataset = train_dataset.map( formatting_prompts_func,
    batched=True,)
validation_dataset = validation_dataset.map( formatting_prompts_func,
    batched=True,)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/685 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

**Train using TRL**


In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, #is true hwen we have short independent incomp sentences
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # Set this for 1 full training run.

        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/685 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/80 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
1.545 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 685 | Num Epochs = 3 | Total steps = 258
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,2.815700
10,2.569900
15,2.297900
20,1.979900
25,1.869800
30,1.931300
35,1.751200
40,1.766100
45,1.694200
50,1.691700


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1202.1564 seconds used for training.
20.04 minutes used for training.
Peak reserved memory = 3.674 GB.
Peak reserved memory for training = 2.129 GB.
Peak reserved memory % of max memory = 25.228 %.
Peak reserved memory for training % of max memory = 14.619 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input and test

In [ ]:
FastLanguageModel.for_inference(model)


messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT,
    },
    {
        "role": "user",
        "content": (
            "Task:\nExplain the following AI research abstract in simple Hinglish\n\n"
            "Research Paper Context:\n"
            "Title: Patient-Specific Articulated Digital Twins from a Single Full-Body CT Scan \nAbstract: Patient-specific anatomical models provide individualized context for surgical planning, image-guided intervention, and algorithm development. However, most CT-derived models are static: they preserve the body configuration captured at scan time, but cannot represent how the same anatomy would appear after patient repositioning. This limitation is especially important for radiographic imaging, where appearance depends jointly on imaging geometry and patient pose. We present a proof-of-concept for constructing a patient-specific articulated digital twin from a single full-body CT scan. The method fits a parametric human body model (SMPL) to obtain a patient-aligned kinematic scaffold, binds segmented bones and organs to an anatomy-aware rig, and retargets body-pose changes while preserving skeletal geometry. On three full-body CT subjects, the fitted scaffold achieved 15.8 $\\pm$ 4.0 mm chamfer distance and 95.9 $\\pm$ 1.8% skeletal enclosure. Recomposition at the acquisition pose preserved major radiographic structure, with overall SSIM of 0.872 $\\pm$ 0.016 and PSNR of 18.5 $\\pm$ 1.4 dB across paired DRRs. Across unseen target poses, the resulting twins enabled articulation while maintaining high skeletal enclosure (94.4 $\\pm$ 0.4%). As a feasibility demonstration, we render the articulated twin as pose-dependent DRRs. These results suggest the feasibility of extending static, view-controllable CT simulation toward pose-controllable anatomical twins for future synthetic imaging and positioning studies."
        ),
    },
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to("cuda")

from transformers import TextStreamer


text_streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)

_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    use_cache=True,
)

Yeh research paper ek naya technique introduce karta hai jo medical imaging ke liye ek specific problem ko solve karta hai. Ab tak humare patients ki internal structures ko samajhne aur prepare karne mein doosre methods kaafi complex thi. Ismein, agar uske paas alag-alag views lene hain, tabhi woh sirf static images dekhne se hi sabko achha nahi hoti. Yeh 'articulated digital twin' concept bahut critical hai kyunki yeh patient ke saath dynamic tarike se interact kar sakta hai.

Is paper mein researchers ne ek smart approach propose kiya hai. Unhone ek 'parametric human body model', jaise SMPL, use kiya tha. Yeh model existing CT scans par fit ho jaata hai, taaki unka alignment precise ho. Phir, unhein segmented bones aur organs ko ek 'anatomy-aware rig' mein bind kiya gaya hai. Isse woh unke position aur spatial relationships ko maintain kar payenge.

Ek bahut bada advantage hai jab hum bone movements ya patient position change karte hue objects ko simulate kar sakte hain. Isse surgeon